# Xử lý dữ liệu

---

## Mục tiêu

Notebook này thực hiện toàn bộ quá trình làm sạch và biến đổi dữ liệu từ 15 file CSV gốc, sau đó xuất ra **4 bộ Master Dataset** phục vụ cho Phần 2 (EDA) và Phần 3 (Forecasting).

**Nguyên tắc xuyên suốt:**
- Không sửa bất cứ gì trong `data/raw/` — toàn bộ output đều được ghi vào `data/processed/`.
- Mọi quyết định xử lý đều có giải thích rõ lý do.
- Giữ nguyên NaN có nghĩa nghiệp vụ, chỉ fill những NaN thực sự cần thiết.

**4 bộ Master Dataset được tạo ra:**

| File | Grain | Mục đích |
|------|-------|----------|
| `sales_master.csv` | 1 sản phẩm / 1 đơn hàng | Phân tích doanh thu, khuyến mãi, hành vi mua theo sản phẩm |
| `customer_master.csv` | 1 khách hàng | Phân tích hành vi khách hàng, RFM, remarketing |
| `ops_master.csv` | 1 đơn hàng | Phân tích vận hành, giao hàng, trả hàng, đánh giá |
| `ops_sales.csv` | 1 đơn hàng (có thông tin sản phẩm đại diện) | Phân tích kết hợp vận hành và sản phẩm — dùng cho Q1, Q2, Q4, Q5, Q12, Q13 trong EDA |

**Cấu trúc notebook:**

| Cell | Nội dung |
|------|----------|
| 1 | Import thư viện và thiết lập đường dẫn |
| 2 | Load 15 file CSV gốc |
| 3 | Làm sạch chung — parse date, chuẩn hóa zip, fill NaN có nghĩa |
| 4 | Xây dựng `sales_master` |
| 5 | Bổ sung time features và Tết features vào `sales_master` |
| 6 | Xây dựng `customer_master` |
| 7 | Xây dựng `ops_master` |
| 8 | Xây dựng `ops_sales` |
| 9 | Kiểm tra chất lượng 4 bộ master |
| 10 | Xuất 4 file master ra `data/processed/` |

---
## Cell 1 — Import thư viện và thiết lập đường dẫn

In [ ]:
import numpy as np
import pandas as pd
import os
from pathlib import Path

# Đường dẫn — chỉnh sửa RAW_DIR nếu chạy trên máy khác
RAW_DIR  = Path('../data/raw')
OUT_DIR  = Path('../data/processed')
OUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)

print(f'Raw dir    : {RAW_DIR.resolve()}')
print(f'Output dir : {OUT_DIR.resolve()}')

**Kết luận Cell 1:** Xác nhận hai đường dẫn in ra đúng với cấu trúc thư mục của máy. Nếu chạy trên Google Colab, thay `RAW_DIR` bằng đường dẫn tương ứng trong Drive.

---
## Cell 2 — Load 15 file CSV gốc

In [ ]:
# Load thuần — không parse_dates, không dtype hint
# Việc parse date và ép kiểu được thực hiện có kiểm soát ở Cell 3
customers         = pd.read_csv(RAW_DIR / 'customers.csv',         low_memory=False)
geography         = pd.read_csv(RAW_DIR / 'geography.csv',         low_memory=False)
inventory         = pd.read_csv(RAW_DIR / 'inventory.csv',         low_memory=False)
order_items       = pd.read_csv(RAW_DIR / 'order_items.csv',       low_memory=False)
orders            = pd.read_csv(RAW_DIR / 'orders.csv',            low_memory=False)
payments          = pd.read_csv(RAW_DIR / 'payments.csv',          low_memory=False)
products          = pd.read_csv(RAW_DIR / 'products.csv',          low_memory=False)
promotions        = pd.read_csv(RAW_DIR / 'promotions.csv',        low_memory=False)
returns           = pd.read_csv(RAW_DIR / 'returns.csv',           low_memory=False)
reviews           = pd.read_csv(RAW_DIR / 'reviews.csv',           low_memory=False)
sales             = pd.read_csv(RAW_DIR / 'sales.csv',             low_memory=False)
sample_submission = pd.read_csv(RAW_DIR / 'sample_submission.csv', low_memory=False)
shipments         = pd.read_csv(RAW_DIR / 'shipments.csv',         low_memory=False)
web_traffic       = pd.read_csv(RAW_DIR / 'web_traffic.csv',       low_memory=False)

# Đăng ký vào dict để tiện xử lý hàng loạt
tables = {
    'customers'  : customers,
    'geography'  : geography,
    'inventory'  : inventory,
    'order_items': order_items,
    'orders'     : orders,
    'payments'   : payments,
    'products'   : products,
    'promotions' : promotions,
    'returns'    : returns,
    'reviews'    : reviews,
    'sales'      : sales,
    'shipments'  : shipments,
    'web_traffic': web_traffic,
}

print(f'Load xong {len(tables)} bảng.')
print()
for name, df in tables.items():
    print(f'  {name:<15} {df.shape[0]:>8,} dòng  x  {df.shape[1]:>2} cột')

**Kết luận Cell 2:** Xác nhận đủ 13 bảng và số dòng khớp với kết quả profiling (ví dụ: `orders` có 646,945 dòng, `order_items` có 714,669 dòng). Nếu con số lệch, kiểm tra lại file gốc trong `data/raw/`.

---
## Cell 3 — Làm sạch chung

Bước này thực hiện 3 thao tác áp dụng cho nhiều bảng:

1. **Parse date** — chuyển tất cả cột date từ `object` sang `datetime64`.
2. **Chuẩn hóa `zip`** — chuyển từ `int64` sang string 5 chữ số (`zfill(5)`) để join nhất quán giữa các bảng.
3. **Fill NaN có nghĩa nghiệp vụ** — chỉ fill những cột mà NaN là do thiếu giá trị thực sự, không phải NaN "by design".

**Lưu ý về `signup_date`:** Profiling phát hiện 477,453 đơn hàng (73.8%) có `order_date < signup_date`. Khả năng cao `signup_date` ghi lại ngày cập nhật hồ sơ gần nhất, không phải ngày đăng ký lần đầu. Không dùng `signup_date` để tính customer tenure hay thời gian từ đăng ký đến mua lần đầu.

In [ ]:
# ------------------------------------------------------------------ #
# 3.1  Parse tất cả cột date một lần
# ------------------------------------------------------------------ #
DATE_COLS_MAP = {
    'customers'  : ['signup_date'],
    'orders'     : ['order_date'],
    'promotions' : ['start_date', 'end_date'],
    'returns'    : ['return_date'],
    'reviews'    : ['review_date'],
    'sales'      : ['Date'],
    'shipments'  : ['ship_date', 'delivery_date'],
    'inventory'  : ['snapshot_date'],
    'web_traffic': ['date'],
}

for table_name, cols in DATE_COLS_MAP.items():
    for col in cols:
        tables[table_name][col] = pd.to_datetime(
            tables[table_name][col], errors='coerce'
        )

print('3.1  Parse date: xong')

# ------------------------------------------------------------------ #
# 3.2  Chuẩn hóa zip: int64 -> string 5 chữ số
#      Lý do: zip 1001 và "01001" sẽ không join được nếu không chuẩn hóa
# ------------------------------------------------------------------ #
for table_name in ['customers', 'geography', 'orders']:
    tables[table_name]['zip'] = (
        tables[table_name]['zip']
        .astype(str)
        .str.zfill(5)
    )

print('3.2  Chuẩn hóa zip: xong')

# ------------------------------------------------------------------ #
# 3.3  Fill NaN có nghĩa nghiệp vụ
#
#  order_items.promo_id / promo_id_2:
#      NaN = đơn không áp dụng khuyến mãi -> fill "NoPromo"
#
#  promotions.applicable_category:
#      NaN = áp dụng tất cả danh mục -> fill "All"
#      (profiling: 40/50 promo là All)
#
#  KHÔNG fill:
#      customers.gender, age_group, acquisition_channel -> nullable by design
#      ops_master.ship_date, return_*, review_*         -> NaN có nghĩa (xem Cell 7)
#      customer_master.total_orders, ...               -> NaN có nghĩa (xem Cell 6)
# ------------------------------------------------------------------ #
order_items['promo_id']   = order_items['promo_id'].fillna('NoPromo')
order_items['promo_id_2'] = order_items['promo_id_2'].fillna('NoPromo')
promotions['applicable_category'] = promotions['applicable_category'].fillna('All')

print('3.3  Fill NaN có nghĩa: xong')

# ------------------------------------------------------------------ #
# 3.4  Drop cột vô nghĩa đã xác định ở profiling
#      inventory.reorder_flag: toàn bộ 60,247 dòng đều = 0, không có thông tin
# ------------------------------------------------------------------ #
if 'reorder_flag' in inventory.columns:
    inventory.drop(columns=['reorder_flag'], inplace=True)

print('3.4  Drop cột vô nghĩa: xong')

**Kết luận Cell 3:** Sau bước này:
- Tất cả cột date đã là `datetime64`, sẵn sàng cho các phép tính thời gian.
- `zip` đồng nhất về format string 5 chữ số trên cả 3 bảng (`customers`, `geography`, `orders`).
- `promo_id`, `promo_id_2`, `applicable_category` đã được fill giá trị mặc định có ý nghĩa.
- `reorder_flag` đã bị xóa khỏi `inventory` vì không mang thông tin.

---
## Cell 4 — Xây dựng `sales_master`

**Grain:** 1 sản phẩm trong 1 đơn hàng (lấy `order_items` làm bảng gốc).

**Quy trình join:**
```
order_items
  LEFT JOIN products    on product_id
  LEFT JOIN orders      on order_id
  LEFT JOIN promotions  on promo_id
```

**Lý do dùng LEFT JOIN:** Giữ nguyên toàn bộ dòng từ `order_items` — không để mất dữ liệu vì thiếu match ở bảng phụ.

**Lý do không join `payments`:** Grain là order item — một đơn có nhiều dòng. Nếu đưa `payment_value` vào, giá trị thanh toán của toàn bộ đơn hàng sẽ bị lặp trên từng sản phẩm, gây sai lệch nghiêm trọng khi tính tổng doanh thu. `payments` được giữ riêng trong `ops_master`.

In [ ]:
# Bước 1: order_items + products
sales_master = order_items.merge(
    products[['product_id', 'product_name', 'category',
              'segment', 'size', 'color', 'price', 'cogs']],
    on='product_id',
    how='left'
)

# Bước 2: + orders
sales_master = sales_master.merge(
    orders[['order_id', 'order_date', 'customer_id',
            'order_status', 'payment_method',
            'device_type', 'order_source', 'zip']],
    on='order_id',
    how='left'
)

# Bước 3: + promotions (chỉ lấy thông tin của promo_id chính)
#          Không join promo_id_2 vì chỉ có 206 dòng, ảnh hưởng không đáng kể
sales_master = sales_master.merge(
    promotions[['promo_id', 'promo_type', 'discount_value',
                'promo_channel', 'stackable_flag',
                'applicable_category', 'min_order_value']],
    on='promo_id',
    how='left'
)

# Fill NaN sinh ra sau khi join promotions:
#   Những dòng promo_id = "NoPromo" sẽ không match -> sinh NaN ở cột của promotions
sales_master['promo_type']          = sales_master['promo_type'].fillna('NoPromo')
sales_master['discount_value']      = sales_master['discount_value'].fillna(0)
sales_master['promo_channel']       = sales_master['promo_channel'].fillna('NoPromo')
sales_master['stackable_flag']      = sales_master['stackable_flag'].fillna(0).astype(int)
sales_master['applicable_category'] = sales_master['applicable_category'].fillna('NoPromo')
sales_master['min_order_value']     = sales_master['min_order_value'].fillna(0)

# Tính thêm cột doanh thu thuần và lợi nhuận gộp theo dòng
sales_master['line_revenue'] = (
    sales_master['unit_price'] * sales_master['quantity']
    - sales_master['discount_amount']
)
sales_master['line_cogs']    = sales_master['cogs'] * sales_master['quantity']
sales_master['line_profit']  = sales_master['line_revenue'] - sales_master['line_cogs']

print(f'sales_master shape: {sales_master.shape}')
print(f'\nNull counts sau khi build:')
null_check = sales_master.isnull().sum()
print(null_check[null_check > 0] if null_check.any() else '  Không có null')

**Kết luận Cell 4:** `sales_master` có grain 1 sản phẩm / 1 đơn hàng. Số dòng bằng số dòng của `order_items` (714,669). Các cột `line_revenue`, `line_cogs`, `line_profit` được tính sẵn để DA không phải tính lại mỗi lần phân tích.

---
## Cell 5 — Bổ sung time features và Tết features vào `sales_master`

Bộ dữ liệu trải dài 2012–2022 tại thị trường Việt Nam. Hành vi mua sắm chịu ảnh hưởng mạnh bởi:
- **Tết Nguyên Đán** — ngày lễ theo lịch âm, không thể dùng `month == 1` hay `month == 2` vì ngày Tết thay đổi mỗi năm. Dùng thư viện `lunardate` để tính chính xác ngày mùng 1 Tết từng năm.
- **Mega Sale** — các ngày 9/9, 10/10, 11/11, 12/12 là ngày siêu khuyến mãi của sàn thương mại điện tử.
- **Cuối tuần** và **quý** — chu kỳ mua sắm thông thường.

In [ ]:
!pip install lunardate -q

from lunardate import LunarDate

# Tính ngày mùng 1 Tết (âm lịch) cho từng năm
tet_map = {}
for year in range(2012, 2025):
    tet_map[year] = pd.Timestamp(
        LunarDate(year, 1, 1).toSolarDate()
    )

print('Ngày Tết các năm trong bộ dữ liệu:')
for yr in range(2012, 2023):
    print(f'  {yr}: {tet_map[yr].date()}')

def add_time_features(df, date_col='order_date'):
    """Thêm các đặc trưng thời gian vào DataFrame.

    Parameters
    ----------
    df       : DataFrame có cột date_col kiểu datetime
    date_col : tên cột date làm gốc

    Returns
    -------
    DataFrame với các cột mới được thêm vào
    """
    df = df.copy()
    dt = df[date_col]

    # Đặc trưng thời gian cơ bản
    df['year']       = dt.dt.year
    df['month']      = dt.dt.month
    df['day']        = dt.dt.day
    df['dayofweek']  = dt.dt.dayofweek   # 0 = Thứ Hai, 6 = Chủ Nhật
    df['quarter']    = dt.dt.quarter
    df['is_weekend'] = (dt.dt.dayofweek >= 5).astype(int)

    # Đặc trưng Tết Nguyên Đán
    df['tet_date']    = df['year'].map(tet_map)
    df['days_to_tet'] = (dt - df['tet_date']).dt.days
    # is_pre_tet  : 20 ngày trước Tết (giai đoạn mua sắm cao điểm)
    df['is_pre_tet']  = df['days_to_tet'].between(-20, -1).astype(int)
    # is_tet_week : tuần diễn ra Tết (mùng 1 đến mùng 7)
    df['is_tet_week'] = df['days_to_tet'].between(0, 7).astype(int)
    # is_post_tet : giai đoạn sau Tết (ngày 8 đến ngày 22)
    df['is_post_tet'] = df['days_to_tet'].between(8, 22).astype(int)
    df = df.drop(columns=['tet_date'])

    # Đặc trưng Mega Sale (các ngày siêu khuyến mãi của sàn TMĐT)
    mega_sale_days = {(9, 9), (10, 10), (11, 11), (12, 12)}
    df['is_mega_sale'] = df.apply(
        lambda r: 1 if (r['month'], r['day']) in mega_sale_days else 0,
        axis=1
    )

    return df

sales_master = add_time_features(sales_master, date_col='order_date')

print(f'\nsales_master shape sau khi thêm time features: {sales_master.shape}')
print('\nCác cột time features đã thêm:')
time_cols = ['year', 'month', 'day', 'dayofweek', 'quarter',
             'is_weekend', 'days_to_tet', 'is_pre_tet',
             'is_tet_week', 'is_post_tet', 'is_mega_sale']
print(sales_master[time_cols].describe().T[['min', 'max', 'mean']].round(3))

**Kết luận Cell 5:** `sales_master` đã có đầy đủ time features. Đặc biệt các cột `is_pre_tet`, `is_tet_week`, `is_mega_sale` là những feature quan trọng cho model forecasting ở Phần 3 — nếu bỏ qua, model sẽ không capture được các đỉnh doanh thu theo mùa vụ đặc thù của thị trường Việt Nam.

---
## Cell 6 — Xây dựng `customer_master`

**Grain:** 1 khách hàng.

**Quy trình join:**
```
customers
  LEFT JOIN geography         on zip          (lấy region, district)
  LEFT JOIN orders aggregated on customer_id  (hành vi đặt hàng)
  LEFT JOIN revenue aggregated on customer_id (doanh thu đã giao)
```

**Lý do giữ NaN sau join:** Profiling phát hiện 31,684 khách hàng chưa từng đặt đơn nào. Đây là insight quan trọng (khách đăng ký nhưng chưa mua) — giữ NaN thay vì fillna(0) để phân biệt "chưa có đơn" với "có đơn nhưng giá trị bằng 0". NaN ở đây phục vụ phân tích remarketing và activation rate.

In [ ]:
# Bước 1: customers + geography (lấy region, district theo zip của khách hàng)
customer_master = customers.merge(
    geography[['zip', 'region', 'district']],
    on='zip',
    how='left'
)

# Bước 2: Aggregate orders theo customer_id
#          Tính tổng hợp hành vi đặt hàng — phải aggregate TRƯỚC khi join
#          vì quan hệ customers:orders là 1:nhiều
customer_orders = orders.groupby('customer_id').agg(
    total_orders     = ('order_id',        'count'),
    first_order_date = ('order_date',      'min'),
    last_order_date  = ('order_date',      'max'),
    fav_device       = ('device_type',     lambda x: x.mode()[0]),
    fav_order_source = ('order_source',    lambda x: x.mode()[0]),
    fav_payment      = ('payment_method',  lambda x: x.mode()[0]),
).reset_index()

# Bước 3: Tính tổng doanh thu và số lượng sản phẩm từ đơn đã giao
#          Chỉ tính đơn delivered — loại cancelled, returned, paid, created
delivered_items = sales_master[sales_master['order_status'] == 'delivered']
customer_revenue = delivered_items.groupby('customer_id').agg(
    total_revenue          = ('line_revenue', 'sum'),
    total_items_sold       = ('quantity',     'sum'),
    total_orders_delivered = ('order_id',     'nunique'),
).reset_index()

# Bước 4: Join tất cả vào customer_master
#          Giữ nguyên NaN cho khách chưa có đơn — không fillna(0)
customer_master = (
    customer_master
    .merge(customer_orders,  on='customer_id', how='left')
    .merge(customer_revenue, on='customer_id', how='left')
)

print(f'customer_master shape: {customer_master.shape}')
print(f'\nNull counts (NaN có nghĩa — khách chưa có đơn hàng):')
null_check = customer_master.isnull().sum()
print(null_check[null_check > 0])
print()
n_no_order = customer_master['total_orders'].isna().sum()
pct = n_no_order / len(customer_master) * 100
print(f'Khách hàng chưa có đơn nào: {n_no_order:,} ({pct:.1f}%) -> giữ NaN')

**Kết luận Cell 6:** `customer_master` có grain 1 khách hàng, tổng số dòng bằng số dòng của `customers` (121,930). NaN trong các cột `total_orders`, `first_order_date`, `total_revenue`... là có nghĩa — tương ứng với khách hàng đã đăng ký nhưng chưa đặt đơn nào. Khi DA lọc phân tích chỉ trên khách có đơn, dùng `customer_master.dropna(subset=['total_orders'])`.

---
## Cell 7 — Xây dựng `ops_master`

**Grain:** 1 đơn hàng.

**Quy trình join:**
```
orders
  LEFT JOIN shipments           on order_id
  LEFT JOIN payments            on order_id
  LEFT JOIN returns aggregated  on order_id
  LEFT JOIN reviews aggregated  on order_id
  LEFT JOIN geography           on zip
```

**Lý do giữ NaN trong `ops_master`:** Mỗi loại NaN đều có ý nghĩa nghiệp vụ rõ ràng:
- `ship_date`, `delivery_date` = NaN: đơn bị hủy hoặc đang chờ xử lý (`cancelled`, `paid`, `created`).
- `return_*` = NaN: đơn không phát sinh trả hàng.
- `review_*` = NaN: đơn chưa được đánh giá.

**Lưu ý về `zip`:** `orders.zip` là địa chỉ **giao hàng**, không phải địa chỉ của khách hàng. Region và district trong `ops_master` phản ánh vùng nhận hàng.

In [ ]:
# Bước 1: Aggregate returns theo order_id
#          returns có quan hệ 1:nhiều với orders -> phải aggregate TRƯỚC khi join
returns_agg = returns.groupby('order_id').agg(
    total_return_qty   = ('return_quantity', 'sum'),
    total_refund       = ('refund_amount',   'sum'),
    return_count       = ('return_id',       'count'),
    main_return_reason = ('return_reason',   lambda x: x.mode()[0]),
).reset_index()

# Bước 2: Aggregate reviews theo order_id
#          reviews có quan hệ 1:nhiều với orders -> phải aggregate TRƯỚC khi join
reviews_agg = reviews.groupby('order_id').agg(
    avg_rating   = ('rating',    'mean'),
    review_count = ('review_id', 'count'),
).reset_index()

# Bước 3: Join tất cả lại
#          Ghi chú: zip từ orders là địa chỉ giao hàng, không phải địa chỉ khách hàng
ops_master = (
    orders
    .merge(shipments,   on='order_id', how='left')
    .merge(payments[['order_id', 'payment_value', 'installments']],
           on='order_id', how='left')
    .merge(returns_agg, on='order_id', how='left')
    .merge(reviews_agg, on='order_id', how='left')
    .merge(
        geography[['zip', 'region', 'district']],
        on='zip',
        how='left'
    )
)

# Tính delivery_days (số ngày từ đặt hàng đến giao hàng)
# Parse lại cả 3 cột date sau merge để tránh lỗi kiểu object
ops_master['order_date']    = pd.to_datetime(ops_master['order_date'],    errors='coerce')
ops_master['ship_date']     = pd.to_datetime(ops_master['ship_date'],     errors='coerce')
ops_master['delivery_date'] = pd.to_datetime(ops_master['delivery_date'], errors='coerce')
ops_master['delivery_days'] = (
    ops_master['delivery_date'] - ops_master['order_date']
).dt.days

print(f'ops_master shape: {ops_master.shape}')
print(f'\nNull counts (NaN có nghĩa — xem ghi chú trong đề bài):')
null_check = ops_master.isnull().sum()
null_show  = null_check[null_check > 0]
print(null_show.to_string())
print()

# Minh họa ý nghĩa NaN theo order_status
print('Phân phối NaN của ship_date theo order_status:')
print(
    ops_master.groupby('order_status')['ship_date']
    .apply(lambda x: x.isna().sum())
    .rename('n_null_ship_date')
    .to_string()
)

**Kết luận Cell 7:** `ops_master` có grain 1 đơn hàng, tổng số dòng bằng số dòng của `orders` (646,945). NaN trong `ship_date`, `delivery_date` tập trung ở các status `cancelled`, `paid`, `created` — đúng với logic nghiệp vụ. Cột `delivery_days` được tính sẵn để phân tích hiệu suất giao hàng. `payments` đã được join vào, cung cấp `payment_value` và `installments` cho phân tích giá trị đơn hàng.

---
## Cell 8 — Xây dựng `ops_sales`

**Grain:** 1 đơn hàng (giữ nguyên grain của `ops_master`).

**Lý do cần file này:** Khi phân tích EDA ở Phần 2, nhiều câu hỏi cần kết hợp thông tin vận hành (`ops_master`) với thông tin sản phẩm như `category`, `segment`, `size`, `net_revenue` — ví dụ:
- Q1: KPI tổng quan cần `net_revenue` từ sản phẩm
- Q2: Trả hàng theo danh mục cần `category`
- Q4: Return rate theo size cần `size`
- Q12: Impact matrix cần `net_revenue` để tính refund/revenue ratio
- Q13: RFM cần `net_revenue` theo customer

Nếu không có `ops_sales`, DA phải tự viết đoạn merge này mỗi lần phân tích, dễ dẫn đến lỗi vì `sales_master` có grain khác (1 sản phẩm / đơn), cần `drop_duplicates('order_id')` đúng cách trước khi join.

**Quy trình build:**
```
ops_master
  LEFT JOIN sales_master (aggregated theo order_id) on order_id
```

Aggregate `sales_master` theo `order_id` trước khi join để:
- Giữ nguyên grain 1 đơn hàng — không bị nhân bản dòng
- Lấy `category`, `segment`, `size` đại diện của đơn (mode — giá trị xuất hiện nhiều nhất)
- Tính `net_revenue` thực của đơn = tổng `line_revenue` của tất cả sản phẩm trong đơn

In [ ]:
# Bước 1: Aggregate sales_master theo order_id
#          Mục tiêu: đưa thông tin sản phẩm về grain 1 đơn hàng
#          - net_revenue: tổng doanh thu thuần của đơn
#          - category, segment, size: giá trị đại diện (mode) của đơn
#            ví dụ: đơn có 3 sản phẩm Streetwear và 1 Outdoor -> category = Streetwear
sales_per_order = sales_master.groupby('order_id').agg(
    net_revenue = ('line_revenue', 'sum'),
    total_qty   = ('quantity',     'sum'),
    category    = ('category',     lambda x: x.mode()[0] if len(x) > 0 else np.nan),
    segment     = ('segment',      lambda x: x.mode()[0] if len(x) > 0 else np.nan),
    size        = ('size',         lambda x: x.mode()[0] if len(x) > 0 else np.nan),
).reset_index()

# Bước 2: Join vào ops_master
#          ops_master đã có grain 1 đơn hàng -> join thẳng, grain không bị vỡ
ops_sales = ops_master.merge(
    sales_per_order,
    on='order_id',
    how='left'
)

print(f'ops_sales shape: {ops_sales.shape}')
print(f'ops_master shape (ref): {ops_master.shape}')
print(f'Grain giữ nguyên: {ops_sales.shape[0] == ops_master.shape[0]}')
print(f'\nNull counts sau khi build:')
null_check = ops_sales[['net_revenue', 'category', 'segment', 'size']].isnull().sum()
print(null_check)
print(f'\nCác cột mới thêm vào so với ops_master:')
new_cols = [c for c in ops_sales.columns if c not in ops_master.columns]
print(new_cols)

**Kết luận Cell 8:** `ops_sales` có grain 1 đơn hàng, số dòng bằng đúng `ops_master` (646,945) — grain không bị vỡ. So với `ops_master`, file này có thêm 4 cột: `net_revenue`, `total_qty`, `category`, `segment`, `size`.

**Khi nào dùng file nào:**
- Dùng `ops_master` khi chỉ cần thông tin vận hành thuần túy (shipment, payment, return, review).
- Dùng `ops_sales` khi cần kết hợp vận hành với thông tin sản phẩm (ví dụ: return rate theo category, revenue theo status).
- Dùng `sales_master` khi cần chi tiết từng sản phẩm trong đơn (ví dụ: doanh thu theo SKU, phân tích khuyến mãi theo sản phẩm).

---
## Cell 9 — Kiểm tra chất lượng 4 bộ master

Xác nhận các điều kiện cơ bản trước khi xuất file:
- Số dòng bằng bảng gốc tương ứng (grain không bị vỡ).
- Không có duplicate theo key chính.
- Các cột tính toán không có giá trị âm bất thường.

In [ ]:
def quality_check(df, name, pk_cols, ref_rows, numeric_check_cols=None):
    """Kiểm tra chất lượng cơ bản của một DataFrame."""
    print(f'=== {name} ===')
    print(f'  Shape              : {df.shape}')

    # Kiểm tra số dòng so với bảng gốc
    row_ok = df.shape[0] == ref_rows
    print(f'  Row count == {ref_rows:,} : {"OK" if row_ok else "FAIL"}')

    # Kiểm tra duplicate theo primary key
    n_dup = df.duplicated(subset=pk_cols).sum()
    print(f'  Duplicates on PK   : {n_dup:,} ({"OK" if n_dup == 0 else "FAIL"})')

    # Kiểm tra giá trị âm trong cột số cần kiểm tra
    if numeric_check_cols:
        for col in numeric_check_cols:
            if col in df.columns:
                n_neg = (df[col] < 0).sum()
                print(f'  {col:<25} < 0 : {n_neg:,} ({"OK" if n_neg == 0 else "CAUTION"})')

    # Tóm tắt null
    null_total = df.isnull().sum().sum()
    print(f'  Total NaN cells    : {null_total:,}')
    print()

quality_check(
    sales_master, 'sales_master',
    pk_cols=['order_id', 'product_id'],
    ref_rows=len(order_items),
    numeric_check_cols=['line_revenue', 'line_cogs', 'line_profit']
)

quality_check(
    customer_master, 'customer_master',
    pk_cols=['customer_id'],
    ref_rows=len(customers),
    numeric_check_cols=['total_revenue', 'total_orders']
)

quality_check(
    ops_master, 'ops_master',
    pk_cols=['order_id'],
    ref_rows=len(orders),
    numeric_check_cols=['payment_value', 'total_refund', 'delivery_days']
)

quality_check(
    ops_sales, 'ops_sales',
    pk_cols=['order_id'],
    ref_rows=len(orders),
    numeric_check_cols=['net_revenue', 'total_refund', 'payment_value']
)

print('Tat ca kiem tra hoan tat.')

**Kết luận Cell 9:** Cả 4 bộ master đều phải pass các điều kiện:
- `Row count == ref_rows : OK` — grain không bị vỡ.
- `Duplicates on PK : 0` — không có dòng trùng.
- Các cột tính toán không có giá trị âm.

Nếu bất kỳ check nào FAIL, dừng lại và điều tra trước khi xuất file.

---
## Cell 10 — Xuất 4 file master ra `data/processed/`

In [ ]:
# Chỉ xuất 4 bộ master — KHÔNG ghi đè lên raw data
# Toàn bộ file gốc trong data/raw/ được giữ nguyên
MASTER_FILES = {
    'sales_master.csv'    : sales_master,
    'customer_master.csv' : customer_master,
    'ops_master.csv'      : ops_master,
    'ops_sales.csv'       : ops_sales,
}

for filename, df in MASTER_FILES.items():
    out_path = OUT_DIR / filename
    df.to_csv(out_path, index=False, encoding='utf-8-sig')
    size_mb = out_path.stat().st_size / 1024 / 1024
    print(f'Xuat: {filename:<25} {df.shape[0]:>9,} dong x {df.shape[1]:>3} cot  |  {size_mb:.1f} MB')

print()
print(f'Output directory: {OUT_DIR.resolve()}')
print(f'Tong: {len(MASTER_FILES)} file master da duoc xuat thanh cong.')

**Kết luận Cell 10:** 4 file master đã được xuất vào `data/processed/`. Cách load cho các thành viên trong nhóm:

```python
import pandas as pd
from pathlib import Path

PROCESSED = Path('../data/processed')

sales_master    = pd.read_csv(PROCESSED / 'sales_master.csv',
                              parse_dates=['order_date'])
customer_master = pd.read_csv(PROCESSED / 'customer_master.csv',
                              parse_dates=['signup_date', 'first_order_date', 'last_order_date'])
ops_master      = pd.read_csv(PROCESSED / 'ops_master.csv',
                              parse_dates=['order_date', 'ship_date', 'delivery_date'])
ops_sales       = pd.read_csv(PROCESSED / 'ops_sales.csv',
                              parse_dates=['order_date', 'ship_date', 'delivery_date'])
```

**Lưu ý quan trọng khi dùng các bộ master:**

| Bộ dữ liệu | Cảnh báo |
|------------|----------|
| `sales_master` | Không tính tổng `payment_value` từ đây — giá trị sẽ bị nhân bản theo số sản phẩm. Dùng `ops_master` hoặc `ops_sales` để tính tổng thanh toán. |
| `customer_master` | NaN trong `total_orders`, `total_revenue`... là khách chưa có đơn — không phải lỗi. |
| `ops_master` | NaN trong `ship_date`, `delivery_date` là đơn chưa được giao. `region` và `district` là của địa chỉ **giao hàng**, không phải địa chỉ khách hàng. |
| `ops_sales` | `category`, `segment`, `size` là giá trị đại diện (mode) của đơn — đơn có nhiều loại sản phẩm sẽ lấy loại xuất hiện nhiều nhất. |
| Mọi bộ | Không dùng `signup_date` để tính customer tenure — cột này không đáng tin (73.8% đơn có `order_date < signup_date`). |

---
## Tổng kết — Xử lý dữ liệu

Notebook này đã thực hiện đầy đủ 10 bước xử lý từ raw data đến 4 bộ Master Dataset sạch và có cấu trúc rõ ràng.

**Tóm tắt những thay đổi so với file Clean_data.ipynb gốc:**

| Thay đổi | Lý do |
|----------|-------|
| Bỏ `quick_explore()` và sanity checks | Đã thực hiện đầy đủ hơn ở `02_data_profiling.ipynb` |
| Sửa output path từ thư mục gốc sang `data/processed/` | Tránh ghi đè raw data |
| Chỉ xuất 4 file master, không xuất lại raw | Raw data phải được bảo toàn nguyên vẹn |
| Thêm `line_revenue`, `line_cogs`, `line_profit` vào `sales_master` | DA không phải tính lại mỗi lần dùng |
| Hoàn thiện `total_revenue` và `total_items_sold` trong `customer_master` | Code gốc tạo `delivered_orders` nhưng không join |
| Thêm `payments` vào `ops_master` | Cung cấp `payment_value`, `installments` cho phân tích |
| Thêm `delivery_days` vào `ops_master` | Feature hữu ích cho phân tích hiệu suất giao hàng |
| Parse lại date sau merge trong `ops_master` | Tránh lỗi `TypeError: cannot subtract DatetimeArray from ndarray[object]` |
| Drop `reorder_flag` khỏi `inventory` | Toàn bộ 60,247 dòng đều = 0, không mang thông tin |
| Thêm `ops_sales` — bộ master thứ 4 | EDA cần join `ops_master` với thông tin sản phẩm ở Q1, Q2, Q4, Q5, Q12, Q13 — tạo sẵn để DA không phải tự join và dễ xảy ra lỗi grain |
| Thêm Cell 9 — quality check 4 file | Xác nhận grain và tính đúng trước khi xuất |
| Ghi chú rõ ý nghĩa NaN và cảnh báo `signup_date` | Phòng ngừa phân tích sai cho DA và MLE |